← [Capstone · Recovering Kepler-8 b](kepler8b_transit_recovery.ipynb) · [Index](README.md)
<!--nav-->

# 06 · Designing a search — and knowing what you can't see

Everything up to here worked on a star whose answer was already published. This notebook
is about the step after that: running a search where **nobody knows the answer**, and
producing a result you can defend.

There is one idea at the centre of it, and it is the difference between a hobby project
and actual science:

> **A search that finds nothing has told you nothing — until you know what it was capable
> of finding.**

Run your pipeline over 500 stars, find no planets, and you cannot distinguish *"there are
no planets here"* from *"my pipeline cannot detect the planets that are here."* Those are
completely different claims about the universe, and nothing in the search itself separates
them.

The fix is **injection and recovery**: put transits you designed into real data, run your
whole pipeline, and count how many come back. That gives you a *completeness map* — and
with it, a null result becomes a real statement: "there are no planets here deeper than X
at periods shorter than Y."

**You'll learn:** how to scope a sample · why you characterise before you search · how to
build and read a completeness map · how to cross-match against what's already known · and
where a hobbyist actually reports something.

## The process, end to end

```
Define a sample  ← NOT random. Write the selection criteria down.
        │
        ▼
Injection–recovery FIRST  ← this notebook
  Measure completeness before you look at anything real.
        │
        ▼
Run the pipeline: detrend → BLS wide scan → TLS on the peaks
        │
        ▼
Automated cuts ──fail──▶ discard (log why, always)
  SDE threshold · odd/even · secondary at phase 0.5
  Rp/Rs < 0.2 · duration consistent with period
        │ survivors
        ▼
Cross-match archives ──known──▶ discard
  NASA Exoplanet Archive · TOI list · Kepler EB Catalog · SIMBAD · Gaia
        │ unknown
        ▼
Manual inspection  ← the irreducibly human step
  fold · pixel-level centroid · persists across sectors?
  present in neighbouring stars? (→ systematic, not sky)
        │
        ▼
Report: ExoFOP-TESS · TFOP · Planet Hunters TESS
```

Two steps people skip, both fatal to the result: **characterising before searching**, and
**logging every rejection**. Without the log you cannot state your selection function, and
without that you don't have a result — you have an anecdote.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from skyplay import data, detrend, injection, models, periods, plotting

plotting.use_style()

## 1. Define a sample

The instinct is "grab random stars." Don't. A random sample makes it impossible to say
anything afterwards, because you cannot describe what you looked at.

Write down criteria instead, and keep them. Something like:

| Criterion | Example | Why |
|---|---|---|
| Mission & cadence | TESS FFI, 30-min | Less exhaustively searched than 2-min targets |
| Spectral type | M dwarfs, T_eff 2,500–3,700 K | Small star → same planet gives a deeper transit |
| Distance | < 25 pc | Brighter → better photometry |
| Magnitude | T < 13 | Sets your noise floor |
| Sample size | 100–1,000 stars | Tractable by hand at the vetting stage |

That table *is* your selection function. It bounds every claim you can make: not "M dwarfs
don't have hot Jupiters" but "among 400 M dwarfs within 25 pc observed in sectors 40–50, I
found none, and I was sensitive to depths above X."

**Before spending any time, find out what's already known.** The NASA Exoplanet Archive is
queryable directly from Python through `astroquery`.

In [ ]:
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive as NEA

total = NEA.query_criteria(table='pscomppars', select='count(*)')
print(f"confirmed planets in the archive: {int(total['count(*)'][0]):,}")

# What is already known about our test star?
known = NEA.query_criteria(
    table='pscomppars',
    select='pl_name,pl_orbper,pl_rade,pl_ratror,discoverymethod',
    where="hostname='Kepler-8'",
)
print()
print(known)

## 2. Characterise the pipeline *before* searching

Now the core of the notebook. We need a light curve with **no real transit in it**, because
we are about to ask whether our pipeline finds transits *we* put there. Kepler-8 b's real
8,400 ppm transit would dominate every search and swamp anything we inject.

So we mask it out. `mask_transits` drops the cadences inside the known transit — about 4%
of the data — leaving a curve that behaves like a normal star with no planet.

In [ ]:
target = data.TARGETS['kepler-8']
lc = data.load_stitched('kepler-8')

# Kepler-8 b's ephemeris, recovered independently back in notebook 03.
P_REAL, T0_REAL, DUR_REAL = 3.52238, 131.6930, 0.10

masked = injection.mask_transits(lc, P_REAL, T0_REAL, DUR_REAL)
print(f'{len(lc)} cadences -> {len(masked)} after masking '
      f'({100 * (1 - len(masked) / len(lc)):.1f}% removed)')

before = injection.default_pipeline(lc)
after = injection.default_pipeline(masked)
print(f'\nBLS on the original curve : {before.period:.4f} d  <- the real planet')
print(f'BLS on the masked curve   : {after.period:.4f} d  <- no planet left, just residual noise')

Note that the masked curve still has a *strongest* peak — searches always return
something. That residual is a systematic, not a planet, and it is exactly what a shallow
injected transit has to compete against.

Let's also note the noise we are working against, since it sets what is possible:

In [ ]:
flat, _ = detrend.savgol_flatten(masked, window_days=18.8)
white = np.diff(flat.flux.value).std() / np.sqrt(2)

# Folding beats noise down as ~sqrt(N), where N is the number of in-transit cadences.
n_in_transit = 60   # roughly, for a ~3 h transit at a few days' period over 311 days
print(f'per-point noise        : {white * 1e6:6.0f} ppm')
print(f'after folding ~{n_in_transit} points : {white / np.sqrt(n_in_transit) * 1e6:6.0f} ppm')
print('\n-> so a detection floor somewhere around 100-200 ppm is what we should expect.')
print('   Expecting is not measuring. Measure it.')

## 3. Inject one transit and look at it

Before running a grid, sanity-check a single injection. `inject_transit` multiplies a real
limb-darkened transit model (via `batman`) into the curve.

Two details it handles for you:

- **Multiplication, not addition.** A transit removes a *fraction* of the light.
- **A physical duration.** Give it a period and it derives the duration from the star's
  mean density, because `a/Rs = (G·ρ·P²/3π)^(1/3)` — the orbital distance in stellar radii
  depends only on period and density. That relation also runs backwards, which is why
  transit surveys double as stellar-density surveys.

In [ ]:
for P in (2.0, 4.0, 8.0):
    print(f'P = {P:4.1f} d  ->  a/Rs = {models.a_rs_from_density(P):5.2f}, '
          f'duration = {models.transit_duration(P, 0.016) * 24:.2f} h')

INJ_PERIOD, INJ_DEPTH = 4.0, 2.5e-4    # 250 ppm: near the floor, roughly Neptune-sized
INJ_EPOCH = float(np.nanmin(masked.time.value)) + 1.7

injected = injection.inject_transit(
    masked, period=INJ_PERIOD, epoch=INJ_EPOCH, depth=INJ_DEPTH
)
print(f'\ninjected {INJ_DEPTH * 1e6:.0f} ppm at P = {INJ_PERIOD} d')

In [ ]:
# Can our pipeline find it? This is one cell of the grid we are about to build.
found = injection.default_pipeline(injected)
print(found.summary())
print()
print('recovered:', injection.is_recovered(found, INJ_PERIOD))

In [ ]:
inj_flat, _ = detrend.savgol_flatten(injected, window_days=18.8)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plotting.plot_folded(inj_flat.time.value, inj_flat.flux.value, INJ_PERIOD, INJ_EPOCH,
                     ax=axes[0], phase_window=0.05,
                     title=f'Folded on the injected period ({INJ_DEPTH * 1e6:.0f} ppm)')
plotting.plot_folded(inj_flat.time.value, inj_flat.flux.value, INJ_PERIOD * 1.31, INJ_EPOCH,
                     ax=axes[1], phase_window=0.05, title='Folded on a wrong period')
axes[1].set_ylim(axes[0].get_ylim())
plt.show();

A 250 ppm transit is barely visible even folded, and invisible on the wrong period — which
is the whole reason we have to measure sensitivity rather than judge it by eye.

## 4. The completeness map

Now the grid. For each (period, depth) cell we inject at several random epochs, run the
full pipeline each time, and record the fraction recovered.

Random epochs matter: a transit that happens to fall in a data gap is missed for reasons
that have nothing to do with its depth, and averaging over epochs stops that from
masquerading as insensitivity.

**Cost:** 4 depths × 4 periods × 5 epochs = 80 pipeline runs, roughly 12 seconds.

In [ ]:
rmap = injection.recovery_grid(
    masked,
    periods=(2.0, 4.0, 6.0, 8.0),
    depths=(6e-5, 1.2e-4, 2.5e-4, 5e-4),
    n_trials=5,
    rng=42,
)
print(rmap.summary())

In [ ]:
plotting.plot_recovery_map(rmap)
plt.show();

### Reading it

- **The bottom rows are your blind spot.** At 60 ppm nothing comes back at any period. If a
  60 ppm planet orbits this star, this pipeline would never know.
- **The top row is safe territory.** Above roughly 500 ppm recovery is essentially complete,
  so a null result there is meaningful.
- **The middle is the boundary**, and it is not a flat line in depth — it depends on period.

That last point is why you measure per-target instead of quoting one threshold. This star's
gaps, systematics and masked regions make some periods harder than others.

Now the payoff. Having measured this, the honest form of a null result is:

> *Searched Kepler-8 (transits of the known planet masked) over 1–10 d. No additional
> signal found. Injection–recovery gives >90% completeness above 500 ppm, falling to zero
> below ~100 ppm, so this excludes additional planets deeper than ~500 ppm in that period
> range — and says nothing whatever about shallower ones.*

That is a result. "I looked and found nothing" is not.

## 5. How much to trust a single cell

One trap. With 5 trials, each is worth 20%, so every number in that map is quantised to
20% and carries real sampling error. Before you interpret any individual cell, check
whether it survives more trials.

In [ ]:
for n in (5, 25):
    r = injection.recovery_grid(masked, periods=(2.0, 6.0), depths=(2.5e-4,),
                               n_trials=n, rng=1)
    print(f'  n_trials={n:2d}   2.0 d -> {r.fraction[0, 0] * 100:3.0f}%    '
          f'6.0 d -> {r.fraction[0, 1] * 100:3.0f}%')

print('\n-> The 2 d vs 6 d difference is real and survives more trials.')
print('   The exact value at 2 d does not: 5 trials put it at 40%, 25 trials at ~52%.')
print('   Use a coarse grid to find the shape of the boundary, a fine one to quote numbers.')
print('   Published work uses hundreds of epochs per cell.')

## 6. Cross-matching survivors

Anything that survives your cuts gets checked against what is already known — *before* you
get attached to it. Most stars with an obvious signal already have a paper.

The four places to look:

| Catalogue | What it rules out |
|---|---|
| [NASA Exoplanet Archive](https://exoplanetarchive.ipac.caltech.edu/) | Confirmed and validated planets |
| [ExoFOP-TESS](https://exofop.ipac.caltech.edu/tess/) TOI list | Existing TESS candidates |
| [Kepler Eclipsing Binary Catalog](http://keplerebs.villanova.edu/) | The dominant false positive |
| SIMBAD / Gaia | Variable stars, close companions, bad astrometry |

A cross-match is a query, not a chore — here is the shape of it:

In [ ]:
# Does anything already exist within a small radius of our target's coordinates?
from astropy import units as u
from astropy.coordinates import SkyCoord

coords = SkyCoord.from_name('KIC 6922244')
print(f'KIC 6922244 -> RA {coords.ra.deg:.4f}, Dec {coords.dec.deg:.4f}')

nearby = NEA.query_region(table='pscomppars', coordinates=coords, radius=1 * u.arcmin,
                          select='pl_name,hostname,pl_orbper,pl_ratror')
print()
print(nearby if len(nearby) else 'nothing in the archive within 1 arcmin')
print('\n-> A hit here means your "discovery" is already published. That is the normal')
print('   outcome, and finding out early is a feature.')

## 7. Where a hobbyist actually reports something

There is no single "submit a discovery" button, and that is appropriate — a candidate needs
independent evidence before it means anything. The realistic routes, in ascending order of
commitment:

- **[Planet Hunters TESS](https://www.zooniverse.org/projects/nora-dot-eisner/planet-hunters-tess)** —
  the citizen-science project. Its forum is where unusual light curves get discussed, and
  it has produced real papers with volunteer co-authors. Boyajian's Star was found this way.
- **[ExoFOP-TESS](https://exofop.ipac.caltech.edu/tess/)** — the community clearing house.
  You can register and upload notes or follow-up photometry against a TOI.
- **TFOP (TESS Follow-up Observing Program)** — if you ever get a telescope, seeing-limited
  photometry of candidate hosts is a genuine, wanted contribution: it checks whether a
  signal is on-target or coming from a nearby eclipsing binary.

What makes a report useful is not the candidate — it's the completeness map, the rejection
log, and the vetting you already did.

## Recap

- A null result requires a **completeness map** to mean anything.
- **Inject before detrending.** Detrending is part of your pipeline and it eats signal.
- **Vary the epoch**, or data gaps will look like insensitivity.
- **Mask real transits** before injecting, or the real planet wins every search.
- Recovery depends on **period as well as depth**, per target. Measure, don't assume.
- **Log every rejection.** The log is your selection function, and the selection function
  is what makes a claim a claim.

## Learning resources
- 📄 [Christiansen et al. (2015), Kepler pipeline completeness via injection](https://arxiv.org/abs/1507.06723) — how the professionals do exactly this
- 📄 [Burke et al. (2015), occurrence rates from Kepler](https://arxiv.org/abs/1506.04175) — what completeness is *for*
- 📗 [ExoFOP-TESS](https://exofop.ipac.caltech.edu/tess/) · [TOI list](https://exofop.ipac.caltech.edu/tess/view_toi.php)
- 📗 [astroquery: NASA Exoplanet Archive](https://astroquery.readthedocs.io/en/latest/ipac/nexsci/nasa_exoplanet_archive.html)

**Next:** there is no next notebook. From here it's your own sample, your own map, and your
own rejection log.